# 🌿 Plant Disease Detection — ResNet50 (50 epochs) + YOLOv5 + OOD

**Optimized to run in ~2 hours on Colab T4 GPU**

Features:
- ✅ ResNet50 classifier (50 full epochs, freeze + fine-tune)
- ✅ YOLOv5 detector
- ✅ **OOD detection**: identifies unknown leaves and shows closest trained match
- ✅ Mixed precision, cudnn benchmark, persistent workers

## 1. GPU Check

In [ ]:
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Mem : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠ No GPU! Runtime → Change runtime type → GPU')

## 2. Install Libraries

In [ ]:
!pip install -q ultralytics timm albumentations kaggle
print('✅ Libraries installed')

## 3. Upload kaggle.json

In [ ]:
from google.colab import files
import os
print('Upload your kaggle.json file:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ kaggle.json configured')

## 4. Download PlantVillage Dataset

In [ ]:
import os
print('Downloading PlantVillage dataset...')
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
print('Extracting...')
!unzip -q plantvillage-dataset.zip -d /content/plantvillage
!rm plantvillage-dataset.zip
print('✅ Dataset ready')

## 5. Imports + Hyperparameters (50 epochs, fast settings)

In [ ]:
import os, random, shutil, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from ultralytics import YOLO
import yaml

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_SIZE      = 128
BATCH_SIZE    = 256
NUM_EPOCHS    = 50
LR            = 2e-3
FREEZE_UNTIL  = 10
TRAIN_SPLIT   = 0.70
VAL_SPLIT     = 0.15
MAX_PER_CLASS = 200

OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'✅ Device: {DEVICE} | IMG: {IMG_SIZE} | BATCH: {BATCH_SIZE} | EPOCHS: {NUM_EPOCHS}')

## 6. Discover Classes

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path('/content/plantvillage/plantvillage dataset/color'),
    Path('/content/plantvillage/PlantVillage'),
    Path('/content/plantvillage'),
]
DATA_ROOT = next(p for p in DATA_ROOT_CANDIDATES if p.exists())
for cand in [DATA_ROOT] + list(DATA_ROOT.rglob('*')):
    if cand.is_dir():
        subs = [d for d in cand.iterdir() if d.is_dir()]
        if len(subs) > 10 and any('___' in s.name or '_' in s.name for s in subs):
            DATA_ROOT = cand; break

exts = {'.jpg','.jpeg','.JPG','.JPEG','.png','.PNG'}
classes = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
class_to_idx = {c:i for i,c in enumerate(classes)}
num_classes = len(classes)
counts = [(c, len([f for f in (DATA_ROOT/c).iterdir() if f.suffix in exts])) for c in classes]
count_df = pd.DataFrame(counts, columns=['Class','Count'])
print(f'✅ {num_classes} classes | {count_df["Count"].sum():,} images')
print(f'   Root: {DATA_ROOT}')

## 7. Visualize Class Distribution + Samples

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(num_classes), count_df['Count'].values,
       color=plt.cm.viridis(np.linspace(0, 1, num_classes)))
ax.set_xticks(range(num_classes))
ax.set_xticklabels(count_df['Class'].values, rotation=90, fontsize=7)
ax.set_title('PlantVillage — Images per Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
sample_classes = random.sample(classes, min(18, num_classes))
for ax, cls in zip(axes.flatten(), sample_classes):
    imgs = [f for f in (DATA_ROOT/cls).iterdir() if f.suffix in exts]
    if imgs:
        img = Image.open(random.choice(imgs)).convert('RGB')
        ax.imshow(img)
        ax.set_title(cls.replace('___','\n').replace('_',' '), fontsize=7)
    ax.axis('off')
plt.suptitle('Sample Images', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Dataset & DataLoaders

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE+16, IMG_SIZE+16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class PlantVillageDataset(Dataset):
    def __init__(self, root, class_to_idx, transform=None, max_per_class=None):
        self.transform = transform
        self.samples = []
        for cls, idx in class_to_idx.items():
            cls_dir = Path(root)/cls
            if not cls_dir.exists(): continue
            files_ = [f for f in cls_dir.iterdir() if f.suffix in exts]
            random.shuffle(files_)
            if max_per_class: files_ = files_[:max_per_class]
            for f in files_:
                self.samples.append((str(f), idx))
        random.shuffle(self.samples)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

class TransformSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, indices, transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        path, label = self.base.samples[self.indices[i]]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

full_dataset = PlantVillageDataset(DATA_ROOT, class_to_idx, max_per_class=MAX_PER_CLASS)
n_total = len(full_dataset)
n_train = int(TRAIN_SPLIT*n_total); n_val = int(VAL_SPLIT*n_total)
n_test  = n_total - n_train - n_val
indices = list(range(n_total))
train_idx, val_idx, test_idx = indices[:n_train], indices[n_train:n_train+n_val], indices[n_train+n_val:]

train_ds = TransformSubset(full_dataset, train_idx, train_transform)
val_ds   = TransformSubset(full_dataset, val_idx,   val_test_transform)
test_ds  = TransformSubset(full_dataset, test_idx,  val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)
print(f'Total: {n_total:,} | Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}')

## 9. ResNet50 Model + Feature Extractor

In [ ]:
def build_resnet50(num_classes, pretrained=True):
    weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = resnet50(weights=weights)
    in_feat = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_feat, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    return model

class FeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.backbone = nn.Sequential(*list(model.children())[:-1])
    def forward(self, x):
        return self.backbone(x).flatten(1)

resnet_model = build_resnet50(num_classes).to(DEVICE)
print(f'Params: {sum(p.numel() for p in resnet_model.parameters()):,}')

## 10. Train ResNet50 — 50 epochs (Phase 1 freeze + Phase 2 fine-tune)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = GradScaler()

def train_one_epoch(model, loader, opt):
    model.train()
    tot_loss, correct, total = 0., 0, 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast():
            out = model(images); loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        tot_loss += loss.item()*images.size(0)
        correct += (out.argmax(1)==labels).sum().item(); total += labels.size(0)
    return tot_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss, correct, total = 0., 0, 0
    preds, lbls = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast():
            out = model(images); loss = criterion(out, labels)
        tot_loss += loss.item()*images.size(0)
        p = out.argmax(1)
        correct += (p==labels).sum().item(); total += labels.size(0)
        preds.extend(p.cpu().numpy()); lbls.extend(labels.cpu().numpy())
    return tot_loss/total, correct/total, preds, lbls

history = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]}
best_val_acc, best_path = 0., str(OUTPUT_DIR/'resnet50_best.pth')
total_start = time.time()

for p in resnet_model.parameters(): p.requires_grad = False
for p in resnet_model.fc.parameters(): p.requires_grad = True
opt = optim.AdamW(filter(lambda p: p.requires_grad, resnet_model.parameters()),
                  lr=LR*3, weight_decay=1e-4)
opt2, sched2 = None, None

print('='*72)
print(f' Phase 1: Frozen backbone (epochs 1–{FREEZE_UNTIL})')
print(f' Phase 2: Full fine-tune  (epochs {FREEZE_UNTIL+1}–{NUM_EPOCHS})')
print('='*72)
print(f"{'Ep':>3}|{'TrLoss':>8}|{'TrAcc':>7}|{'VlLoss':>8}|{'VlAcc':>7}|{'Time':>5}|{'ETA':>7}|Phase")
print('-'*72)

for epoch in range(1, NUM_EPOCHS+1):
    if epoch == FREEZE_UNTIL+1:
        print(f'\n🔓 Epoch {epoch}: Unfreezing all layers — Phase 2\n')
        for p in resnet_model.parameters(): p.requires_grad = True
        opt2 = optim.AdamW(resnet_model.parameters(), lr=LR, weight_decay=1e-4)
        sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=NUM_EPOCHS-FREEZE_UNTIL)
        opt = opt2
    t0 = time.time()
    tr_l, tr_a = train_one_epoch(resnet_model, train_loader, opt)
    vl_l, vl_a, _, _ = evaluate(resnet_model, val_loader)
    if epoch > FREEZE_UNTIL and sched2: sched2.step()
    history['train_loss'].append(tr_l); history['train_acc'].append(tr_a)
    history['val_loss'].append(vl_l);   history['val_acc'].append(vl_a)
    elapsed = time.time()-t0
    eta_min = elapsed*(NUM_EPOCHS-epoch)/60
    phase = 'Frz' if epoch<=FREEZE_UNTIL else 'Full'
    tag = ' '
    if vl_a > best_val_acc:
        best_val_acc = vl_a
        torch.save(resnet_model.state_dict(), best_path)
        tag = '★'
    print(f'{epoch:>3}|{tr_l:>8.4f}|{tr_a:>7.4f}|{vl_l:>8.4f}|{vl_a:>7.4f}|{elapsed:>4.0f}s|{eta_min:>6.1f}m|{phase}{tag}')

print(f'\n✅ Training complete in {(time.time()-total_start)/60:.1f} min')
print(f'✅ Best val accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

## 11. Training Curves

In [ ]:
ep = range(1, NUM_EPOCHS+1)
fig, (a1,a2) = plt.subplots(1,2, figsize=(14,5))
a1.plot(ep, history['train_loss'], 'b-o', ms=3, label='Train')
a1.plot(ep, history['val_loss'],   'r-o', ms=3, label='Val')
a1.axvline(FREEZE_UNTIL, color='gray', ls='--', alpha=.7, label=f'Unfreeze (ep {FREEZE_UNTIL})')
a1.set_title('Loss — ResNet50 (50 epochs)', fontweight='bold')
a1.set_xlabel('Epoch'); a1.legend(); a1.grid(alpha=.3)
a2.plot(ep, history['train_acc'], 'b-o', ms=3, label='Train')
a2.plot(ep, history['val_acc'],   'r-o', ms=3, label='Val')
a2.axvline(FREEZE_UNTIL, color='gray', ls='--', alpha=.7, label=f'Unfreeze (ep {FREEZE_UNTIL})')
a2.set_title('Accuracy — ResNet50 (50 epochs)', fontweight='bold')
a2.set_xlabel('Epoch'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'resnet50_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Build OOD Prototype Bank + Calibrate Threshold

In [ ]:
resnet_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
resnet_model.eval()
feat_extractor = FeatureExtractor(resnet_model).to(DEVICE).eval()
FEAT_DIM = 2048

@torch.no_grad()
def compute_prototypes():
    sums = torch.zeros(num_classes, FEAT_DIM, device=DEVICE)
    cnts = torch.zeros(num_classes, device=DEVICE)
    loader = DataLoader(TransformSubset(full_dataset, train_idx, val_test_transform),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    for imgs, lbls in tqdm(loader, desc='Prototypes'):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast():
            f = feat_extractor(imgs)
        f = F.normalize(f.float(), dim=1)
        lbls_d = lbls.to(DEVICE)
        for c in lbls.unique():
            mask = (lbls_d == c.to(DEVICE))
            sums[c] += f[mask].sum(0)
            cnts[c] += mask.sum()
    return F.normalize(sums / cnts.unsqueeze(1).clamp(min=1), dim=1)

prototypes = compute_prototypes()
torch.save(prototypes.cpu(), OUTPUT_DIR/'prototypes.pt')

@torch.no_grad()
def calibrate_threshold():
    sims = []
    loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    for imgs, _ in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast():
            f = F.normalize(feat_extractor(imgs).float(), dim=1)
        s = (f @ prototypes.T).max(1).values
        sims.extend(s.cpu().numpy())
    sims = np.array(sims)
    return float(np.percentile(sims, 5)), sims

OOD_THRESHOLD, val_sims = calibrate_threshold()
print(f'✅ OOD threshold: {OOD_THRESHOLD:.3f}')
print(f'   Val sim mean: {val_sims.mean():.3f} | min: {val_sims.min():.3f}')
json.dump({'threshold': OOD_THRESHOLD, 'img_size': IMG_SIZE},
          open(OUTPUT_DIR/'ood_config.json','w'))

## 13. Test Set Evaluation

In [ ]:
_, test_acc, test_preds, test_labels = evaluate(resnet_model, test_loader)
f1  = f1_score(test_labels, test_preds, average='weighted')
pre = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
rec = recall_score(test_labels, test_preds, average='weighted')
print('='*45)
print('   ResNet50 — Test Set Results')
print('='*45)
print(f'  Accuracy : {test_acc*100:.2f}%')
print(f'  F1-Score : {f1:.4f}')
print(f'  Precision: {pre:.4f}')
print(f'  Recall   : {rec:.4f}')
print('='*45)

In [ ]:
short_labels = [c.replace('___',' / ').replace('_',' ')[:22] for c in classes]
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=short_labels, yticklabels=short_labels,
            linewidths=0.3, ax=ax)
ax.set_title('Confusion Matrix — ResNet50 (50 epochs)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.tick_params(axis='x', labelsize=7, rotation=90)
ax.tick_params(axis='y', labelsize=7, rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
per_df = pd.DataFrame({'Class': short_labels, 'Accuracy': per_class_acc}).sort_values('Accuracy')
fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#2ecc71' if a>=0.9 else '#e67e22' if a>=0.7 else '#e74c3c' for a in per_df['Accuracy']]
ax.barh(per_df['Class'], per_df['Accuracy'], color=colors)
ax.axvline(0.9, color='green',  ls='--', alpha=.7, label='90%')
ax.axvline(0.7, color='orange', ls='--', alpha=.7, label='70%')
ax.set_xlim(0, 1.05); ax.set_title('Per-Class Accuracy', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'≥90%: {(per_class_acc>=0.9).sum()} | <70%: {(per_class_acc<0.7).sum()}')

## 14. Prepare YOLO Dataset

In [ ]:
YOLO_ROOT = Path('/content/yolo_dataset')
for split in ['train','val','test']:
    (YOLO_ROOT/split/'images').mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT/split/'labels').mkdir(parents=True, exist_ok=True)

def write_yolo(img_path, label_idx, split):
    src = Path(img_path)
    shutil.copy(src, YOLO_ROOT/split/'images'/src.name)
    with open(YOLO_ROOT/split/'labels'/src.with_suffix('.txt').name,'w') as f:
        f.write(f'{label_idx} 0.5 0.5 1.0 1.0\n')

YOLO_PER_CLASS = 40
for cls, idx in tqdm(class_to_idx.items(), desc='YOLO data'):
    imgs = [f for f in (DATA_ROOT/cls).iterdir() if f.suffix in exts]
    random.shuffle(imgs); imgs = imgs[:YOLO_PER_CLASS]
    n = len(imgs); tr = int(.7*n); vl = int(.85*n)
    for i, im in enumerate(imgs):
        sp = 'train' if i<tr else ('val' if i<vl else 'test')
        write_yolo(im, idx, sp)

yaml_path = YOLO_ROOT/'data.yaml'
yaml.dump({'path':str(YOLO_ROOT.resolve()),'train':'train/images',
          'val':'val/images','test':'test/images',
          'nc':num_classes,'names':classes},
          open(yaml_path,'w'), sort_keys=False)
print('✅ YOLO data ready')

## 15. Train YOLOv5n (fast)

In [ ]:
yolo_model = YOLO('yolov5n.pt')
yolo_start = time.time()
yolo_results = yolo_model.train(
    data=str(yaml_path), epochs=20, imgsz=224, batch=64,
    lr0=1e-2, cos_lr=True, patience=5, cache=True,
    workers=4, device=0, plots=False, verbose=False,
    name='plant_yolo', project=str(OUTPUT_DIR/'runs'))
print(f'✅ YOLO done in {(time.time()-yolo_start)/60:.1f} min')
best_weights = str(OUTPUT_DIR/'runs/plant_yolo/weights/best.pt')

In [ ]:
yolo_best = YOLO(best_weights)
val_results = yolo_best.val(data=str(yaml_path), split='test', imgsz=224, verbose=False)
map50    = val_results.box.map50
map50_95 = val_results.box.map
print(f'mAP@0.5     : {map50:.4f}')
print(f'mAP@0.5:0.95: {map50_95:.4f}')

## 16. Smart Combined System (YOLO + ResNet50 + OOD)

In [ ]:
class SmartPlantSystem:
    def __init__(self, yolo_w, resnet_w, classes, prototypes, ood_threshold, device):
        self.detector = YOLO(yolo_w)
        self.classifier = build_resnet50(len(classes)).to(device)
        self.classifier.load_state_dict(torch.load(resnet_w, map_location=device))
        self.classifier.eval()
        self.feat_ext = FeatureExtractor(self.classifier).to(device).eval()
        self.classes = classes
        self.prototypes = prototypes.to(device)
        self.threshold = ood_threshold
        self.device = device
        self.tf = val_test_transform

    @torch.no_grad()
    def _classify_crop(self, pil_img):
        t = self.tf(pil_img).unsqueeze(0).to(self.device)
        with autocast():
            logits = self.classifier(t)
            probs  = F.softmax(logits, dim=1)[0]
            feat   = F.normalize(self.feat_ext(t).float(), dim=1)
        top_p, top_i = probs.topk(3)
        sims = (feat @ self.prototypes.T)[0]
        best_sim, best_sim_idx = sims.max(0)
        is_known = best_sim.item() >= self.threshold
        return {
            'top1_class': self.classes[top_i[0].item()],
            'top1_prob' : top_p[0].item(),
            'top3'      : [(self.classes[i.item()], p.item()) for p,i in zip(top_p, top_i)],
            'closest_trained': self.classes[best_sim_idx.item()],
            'similarity': best_sim.item(),
            'is_known'  : is_known,
        }

    @torch.no_grad()
    def predict(self, image_path, conf=0.25):
        img = Image.open(image_path).convert('RGB')
        W, H = img.size
        det = self.detector.predict(image_path, conf=conf, imgsz=224, verbose=False)[0]
        outputs = []
        boxes = det.boxes if len(det.boxes) else None
        if boxes is None:
            r = self._classify_crop(img); r['box']=[0,0,W,H]; r['source']='full-image'
            outputs.append(r)
        else:
            for b in boxes:
                x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
                x1,y1 = max(0,x1), max(0,y1); x2,y2 = min(W,x2), min(H,y2)
                crop = img.crop((x1,y1,x2,y2))
                r = self._classify_crop(crop)
                r['box']=[x1,y1,x2,y2]; r['source']='detect+classify'
                outputs.append(r)
        return img, outputs

    def visualize(self, image_path, conf=0.25, save_path=None):
        img, outs = self.predict(image_path, conf)
        fig, ax = plt.subplots(figsize=(10,7))
        ax.imshow(img)
        colors = plt.cm.Set2.colors
        for i, o in enumerate(outs):
            x1,y1,x2,y2 = o['box']
            color = colors[i % len(colors)]
            ec = 'red' if not o['is_known'] else color
            rect = mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=3, edgecolor=ec, facecolor='none')
            ax.add_patch(rect)
            if o['is_known']:
                short = o['top1_class'].replace('___',' / ').replace('_',' ')
                txt = f"✓ {short}\nconf {o['top1_prob']:.1%} | sim {o['similarity']:.2f}"
                bg = color
            else:
                short = o['closest_trained'].replace('___',' / ').replace('_',' ')
                txt = f"⚠ UNKNOWN LEAF\nclosest: {short}\nsim {o['similarity']:.2f} (thr {self.threshold:.2f})"
                bg = 'red'
            ax.text(x1, max(0,y1-8), txt, color='white', fontsize=9, fontweight='bold',
                    bbox=dict(facecolor=bg, alpha=0.8, pad=3))
        ax.set_title('Plant Disease — ResNet50 (50ep) + YOLO + OOD', fontweight='bold')
        ax.axis('off'); plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=140, bbox_inches='tight')
        plt.show()
        return outs

system = SmartPlantSystem(best_weights, best_path, classes,
                          prototypes, OOD_THRESHOLD, DEVICE)
trained_plants = sorted(set(c.split('___')[0] for c in classes))
print('✅ System ready')
print(f'   Trained plants ({len(trained_plants)}): {trained_plants}')

## 17. Test on Held-out Trained Samples

In [ ]:
test_paths = [full_dataset.samples[i][0] for i in test_idx[:3]]
for p in test_paths:
    print(f'\n📷 {Path(p).name}')
    res = system.visualize(p, save_path=str(OUTPUT_DIR/f'res_{Path(p).stem}.png'))
    for r in res:
        if r['is_known']:
            print(f"  ✓ {r['top1_class']} ({r['top1_prob']:.1%})")
        else:
            print(f"  ⚠ Unknown! Closest: {r['closest_trained']}")

## 18. Upload Your Own Image (Trained or Unknown Plant)

In [ ]:
from google.colab import files
print('Upload a leaf image (any plant — even one NOT in training):')
uploaded = files.upload()
for fname in uploaded:
    print(f'\n📷 Testing: {fname}')
    res = system.visualize(fname, save_path=str(OUTPUT_DIR/f'user_{Path(fname).stem}.png'))
    for r in res:
        if r['is_known']:
            plant = r['top1_class'].split('___')[0].replace('_',' ')
            disease = r['top1_class'].split('___')[-1].replace('_',' ')
            print(f'  ✅ TRAINED PLANT DETECTED')
            print(f'     Plant      : {plant}')
            print(f'     Disease    : {disease}')
            print(f"     Confidence : {r['top1_prob']:.2%}")
            print(f"     Similarity : {r['similarity']:.3f}")
        else:
            closest = r['closest_trained']
            plant = closest.split('___')[0].replace('_',' ')
            disease = closest.split('___')[-1].replace('_',' ')
            print(f'  ⚠️ UNKNOWN LEAF (not in training set)')
            print(f"     Similarity : {r['similarity']:.3f} < threshold {OOD_THRESHOLD:.3f}")
            print(f'     Closest trained plant : {plant}')
            print(f'     Closest trained class : {disease}')
            print(f'     Top-3 closest:')
            for c, p in r['top3']:
                pl = c.split('___')[0].replace('_',' ')
                ds = c.split('___')[-1].replace('_',' ')
                print(f'        • {pl} → {ds}  ({p:.1%})')
            print(f'\n     ℹ️ Trained plants: {trained_plants}')

## 19. Save Results Summary

In [ ]:
summary = pd.DataFrame([
    {'Model':'ResNet50','Task':'Classification','Epochs':50,
     'Accuracy':f'{test_acc*100:.2f}%','F1':f'{f1:.4f}',
     'Precision':f'{pre:.4f}','Recall':f'{rec:.4f}','mAP@0.5':'—'},
    {'Model':'YOLOv5n','Task':'Object Detection','Epochs':20,
     'Accuracy':'—','F1':'—','Precision':'—','Recall':'—',
     'mAP@0.5':f'{map50:.4f}'}
])
print('='*72)
print('   PLANT DISEASE DETECTION — FINAL RESULTS')
print('='*72)
print(summary.to_string(index=False))
summary.to_csv(OUTPUT_DIR/'results_summary.csv', index=False)
report = classification_report(test_labels, test_preds, target_names=short_labels,
                              output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR/'per_class_report.csv')
print('\n✅ CSVs saved')

## 20. Download Results

In [ ]:
import zipfile
zip_path = '/content/plant_disease_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in OUTPUT_DIR.rglob('*'):
        if f.is_file() and f.suffix in ['.png','.csv','.pth','.pt','.yaml','.json']:
            zf.write(f, f.relative_to(OUTPUT_DIR))
print('Zip contents:')
with zipfile.ZipFile(zip_path,'r') as zf:
    for n in sorted(zf.namelist()): print(f'  {n}')
from google.colab import files
files.download(zip_path)
print('\n✅ Download started!')